# ♟️ Chess Move Recognition — Demo Notebook

End-to-end demonstration of the **Automated Chess Move Recognition in Live Video** pipeline.

Steps:
1. Load dependencies & config
2. Calibrate camera (optional)
3. Detect pieces in one image
4. Run full pipeline on video
5. Evaluate & plot metrics

In [6]:
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import yaml

from src.detection.detect_pieces import PieceDetector
from src.evaluation.evaluate import compare, load_moves
from src.inference.pipeline import run_pipeline
from src.utils.visualize import draw_detections

with open('../config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

VIDEO_DIR = Path(cfg['paths']['video_dir'])
RESULTS_DIR = Path(cfg['paths']['results_dir'])
MODEL_PATH = cfg['paths']['model_path']
GROUND_TRUTH_PATH = Path(cfg['paths']['ground_truth'])
RESULTS_DIR.mkdir(exist_ok=True, parents=True)
print(f'Using model: {MODEL_PATH}')

Using model: ../models/yolov8n.pt


In [7]:
# --- Quick test: piece detection ---
detector = PieceDetector(model_path=MODEL_PATH)
sample_image = next(Path('data/processed').rglob('*.jpg'), None)
if sample_image:
    img = cv2.imread(str(sample_image))
    pieces = detector.detect(img)
    vis = draw_detections(img, pieces)
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB));
    plt.axis('off')
else:
    print('No image found for test.')

[INFO] Loading YOLOv8 model from ../models/yolov8n.pt ...
[READY] Model loaded with 80 classes.
No image found for test.


In [8]:
# --- Run pipeline and evaluate ---
video_files = list(VIDEO_DIR.glob('*.mp4'))
if not video_files:
    print('No videos in data/raw/')
else:
    vp = str(video_files[0])
    run_pipeline(vp, MODEL_PATH, str(RESULTS_DIR), save_vis=True)
    pred_path = RESULTS_DIR / f"{Path(vp).stem}_pred.json"
    pred_moves = load_moves(str(pred_path))
    truth_moves = load_moves(str(GROUND_TRUTH_PATH))
    metrics = compare(pred_moves, truth_moves)
    print(json.dumps(metrics, indent=2))

No videos in data/raw/


In [9]:
# --- Plot aggregated metrics ---
import pandas as pd

result_files = list(RESULTS_DIR.glob('*_pred.json'))
rows = []
for rf in result_files:
    pred_moves = load_moves(str(rf))
    truth_moves = load_moves(str(GROUND_TRUTH_PATH))
    m = compare(pred_moves, truth_moves)
    m['file'] = rf.stem
    rows.append(m)

if rows:
    df = pd.DataFrame(rows)
    df[['precision', 'recall']].plot(kind='bar', figsize=(8, 4))
    plt.title('Precision and Recall per Game')
    plt.xlabel('Game Index')
    plt.ylabel('Score')
    plt.ylim(0, 1)
    plt.show()
else:
    print('No result files to plot.')

No result files to plot.
